In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
list_files = os.listdir("../")
list_files

['.git',
 'away_team.csv',
 'away_team_score.csv',
 'event.csv',
 'home_team.csv',
 'home_team_score.csv',
 'Javadi',
 'notebook.ipynb',
 'odds.csv',
 'pbp.csv',
 'power.csv',
 'round.csv',
 'season.csv',
 'statistics.csv',
 'time.csv',
 'tournament.csv',
 'venue.csv',
 'votes.csv']

In [3]:
df_statistics = pd.read_csv("../statistics.csv")
event_df = pd.read_csv("../event.csv")
df_statistics

,match_id,period,statistic_category_name,statistic_name,home_stat,away_stat,compare_code,statistic_type,value_type,home_value,away_value,home_total,away_total
0,11998445,ALL,service,aces,12,6,1,positive,event,12,6,NaN,NaN
1,11998445,ALL,service,double_faults,2,7,2,negative,event,2,7,NaN,NaN
2,11998445,ALL,service,first_serve,57/101 (56%),53/90 (59%),2,positive,team,57,53,101.0,90.0
3,11998445,ALL,service,second_serve,42/44 (95%),30/37 (81%),1,positive,team,42,30,44.0,37.0
4,11998445,ALL,service,first_serve_points,42/57 (74%),39/53 (74%),1,positive,team,42,39,57.0,53.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1358229,12213803,2ND,games,max_games_in_a_row,4,1,1,positive,event,4,1,NaN,NaN
1358230,12213803,2ND,return,first_serve_return_points,10/16 (62%),4/13 (30%),1,positive,team,10,4,16.0,13.0
1358231,12213803,2ND,return,second_serve_return_points,8/14 (57%),6/13 (46%),1,positive,team,8,6,14.0,13.0
1358232,12213803,2ND,return,return_games_played,4,3,1,positive,event,4,3,NaN,NaN


In [4]:

# ─────────────────────────────────────────────────────────────────────────────
# THRESHOLDS — grounded in real tennis records
# Per player per match: 0–120 (buffer above Isner's all-time record of 113)
# Combined (home + away): 0–240
# ─────────────────────────────────────────────────────────────────────────────
MAX_ACES_PER_PLAYER  = 113
MAX_ACES_COMBINED    = 226
MIN_ACES             = 0

report         = []
total_original = len(df_statistics)

def log(step, desc, removed, note=''):
    report.append({
        'Step'           : step,
        'Description'    : desc,
        'Rows Removed'   : removed,
        '% of Original'  : round(removed / total_original * 100, 2),
        'Note'           : note
    })

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Filter to aces rows and period = 'ALL' only
# Keeping per-set periods would double/triple count aces
# ─────────────────────────────────────────────────────────────────────────────
df = df_statistics.copy()
before      = len(df)
df          = df[(df['statistic_name'] == 'aces') & (df['period'] == 'ALL')].copy()
removed     = before - len(df)
log(1, "Kept only statistic_name='aces' AND period='ALL'", removed,
    'Removes all non-ace rows and per-set ace rows to avoid double counting')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Drop exact duplicate rows
# ─────────────────────────────────────────────────────────────────────────────
before  = len(df)
df      = df.drop_duplicates()
removed = before - len(df)
log(2, 'Exact duplicate rows', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Handle duplicate match_ids — corrupt vs interrupted matches
# For statistics, an interrupted match should still have ONE 'ALL' period row
# So any match_id appearing more than once here is corrupt, not interrupted
# ─────────────────────────────────────────────────────────────────────────────
before      = len(df)
dupes       = df[df['match_id'].duplicated(keep=False)]
n_corrupt   = dupes['match_id'].nunique()
df          = df[~df['match_id'].duplicated(keep=False)].copy()
removed     = before - len(df)
log(3, 'Corrupt duplicate match_ids (multiple ALL-period ace rows)',
    removed, f'{n_corrupt} match_ids affected — no interruption logic needed here '
             f'since ALL period should always be exactly one row per match')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Cross-validate home_value vs home_stat string column
# home_stat stores the same value as a string — they must agree
# If they disagree, the row is unreliable; null out home_value/away_value
# ─────────────────────────────────────────────────────────────────────────────
def extract_numeric(series):
    """Safely extract numeric value from potentially string column."""
    return pd.to_numeric(series, errors='coerce')

home_stat_numeric = extract_numeric(df['home_stat'])
away_stat_numeric = extract_numeric(df['away_stat'])

home_mismatch = (
    home_stat_numeric.notna() &
    df['home_value'].notna() &
    (home_stat_numeric != df['home_value'])
)
away_mismatch = (
    away_stat_numeric.notna() &
    df['away_value'].notna() &
    (away_stat_numeric != df['away_value'])
)

cells_nulled = home_mismatch.sum() + away_mismatch.sum()
df.loc[home_mismatch, 'home_value'] = np.nan
df.loc[away_mismatch, 'away_value'] = np.nan

report.append({
    'Step'          : 4,
    'Description'   : 'home_value/away_value nulled where string stat disagrees with numeric value',
    'Rows Removed'  : f'{cells_nulled} cells nulled',
    '% of Original' : round(cells_nulled / total_original * 100, 2),
    'Note'          : 'Uses home_stat/away_stat as ground truth'
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Drop rows where both home_value and away_value are null
# We cannot compute aces for a match without at least one value
# ─────────────────────────────────────────────────────────────────────────────
before  = len(df)
df      = df[df['home_value'].notna() | df['away_value'].notna()].copy()
removed = before - len(df)
log(5, 'Rows where both home_value and away_value are null', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Fill remaining nulls using string stat columns as fallback
# If home_value is null but home_stat has a valid number, use it
# ─────────────────────────────────────────────────────────────────────────────
filled_home = df['home_value'].isna() & home_stat_numeric.notna()
filled_away = df['away_value'].isna() & away_stat_numeric.notna()
df.loc[filled_home, 'home_value'] = home_stat_numeric[filled_home]
df.loc[filled_away, 'away_value'] = away_stat_numeric[filled_away]

report.append({
    'Step'          : 6,
    'Description'   : 'Null home/away values recovered from home_stat/away_stat strings',
    'Rows Removed'  : f'{filled_home.sum() + filled_away.sum()} cells recovered',
    '% of Original' : '-',
    'Note'          : 'Recovery step, not removal'
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: Negative ace counts — physically impossible
# ─────────────────────────────────────────────────────────────────────────────
before      = len(df)
neg_mask    = (df['home_value'] < MIN_ACES) | (df['away_value'] < MIN_ACES)
df          = df[~neg_mask].copy()
removed     = before - len(df)
log(7, 'Rows with negative ace counts', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: Unrealistically high ace counts per player
# Ceiling = 120 per player (buffer above Isner's all-time record of 113)
# ─────────────────────────────────────────────────────────────────────────────
before   = len(df)
high_mask = (
    (df['home_value'] > MAX_ACES_PER_PLAYER) |
    (df['away_value'] > MAX_ACES_PER_PLAYER)
)
df       = df[~high_mask].copy()
removed  = before - len(df)
log(8, f'Rows where a player exceeded {MAX_ACES_PER_PLAYER} aces (above all-time record)',
    removed, 'Isner holds record of 113 in a 5-set match; 47 in a 3-set match')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 9: Unrealistically high combined aces
# ─────────────────────────────────────────────────────────────────────────────
df['total_aces'] = df['home_value'].fillna(0) + df['away_value'].fillna(0)
before    = len(df)
comb_mask = df['total_aces'] > MAX_ACES_COMBINED
df        = df[~comb_mask].copy()
removed   = before - len(df)
log(9, f'Rows where combined aces exceed {MAX_ACES_COMBINED}', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 10: Cross-validate against event_df — only finished matches
# ─────────────────────────────────────────────────────────────────────────────
before    = len(df)
valid_ids = event_df[event_df['winner_code'].notna()]['match_id']
df        = df[df['match_id'].isin(valid_ids)].copy()
removed   = before - len(df)
log(10, 'Matches not in event_df or with null winner_code (unfinished)',
    removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# COMPUTE: aces per match
# home_value = home player aces, away_value = away player aces
# total_aces = combined per match
# ─────────────────────────────────────────────────────────────────────────────
df['total_aces'] = df['home_value'] + df['away_value']

# ─────────────────────────────────────────────────────────────────────────────
# PRINT REPORT
# ─────────────────────────────────────────────────────────────────────────────
total_clean   = len(df)
total_removed = total_original - total_clean

print("=" * 70)
print("DATA CLEANING REPORT — Aces per Match")
print("=" * 70)
print(f"  Original rows       : {total_original:,}")
print(f"  Rows after cleaning : {total_clean:,}")
print(f"  Total removed       : {total_removed:,} ({round(total_removed/total_original*100,2)}%)")
print(f"  Clean data          : {round(total_clean/total_original*100,2)}%")
print("=" * 70)
print()
print(pd.DataFrame(report).to_string(index=False))


DATA CLEANING REPORT — Aces per Match
  Original rows       : 1,358,234
  Rows after cleaning : 10,201
  Total removed       : 1,348,033 (99.25%)
  Clean data          : 0.75%

 Step                                                                 Description        Rows Removed % of Original                                                                                                                    Note
    1                            Kept only statistic_name='aces' AND period='ALL'             1334951         98.29                                                  Removes all non-ace rows and per-set ace rows to avoid double counting
    2                                                        Exact duplicate rows               10701          0.79                                                                                                                        
    3                  Corrupt duplicate match_ids (multiple ALL-period ace rows)                2319          0.17

In [5]:

# ─────────────────────────────────────────────────────────────────────────────
# DESCRIPTIVE STATISTICS — three views:
# 1. Total aces per match (home + away combined)
# 2. Home player aces per match
# 3. Away player aces per match
# ─────────────────────────────────────────────────────────────────────────────
def print_stats(series, label):
    desc = series.describe(percentiles=[0.25, 0.5, 0.75])
    print(f"\n{'=' * 70}")
    print(f"DESCRIPTIVE STATISTICS — {label}")
    print(f"{'=' * 70}")
    print(f"  Count   : {int(desc['count']):,}")
    print(f"  Mean    : {desc['mean']:.4f}")
    print(f"  Std Dev : {desc['std']:.4f}")
    print(f"  Min     : {desc['min']:.0f}")
    print(f"  25%     : {desc['25%']:.2f}")
    print(f"  Median  : {desc['50%']:.2f}")
    print(f"  75%     : {desc['75%']:.2f}")
    print(f"  Max     : {desc['max']:.0f}")
    print(f"  Skew    : {series.skew():.4f}")
    print(f"  Kurt    : {series.kurt():.4f}")

print_stats(df['total_aces'], 'Total Aces per Match (Home + Away)')
print_stats(df['home_value'], 'Home Player Aces per Match')
print_stats(df['away_value'], 'Away Player Aces per Match')


DESCRIPTIVE STATISTICS — Total Aces per Match (Home + Away)
  Count   : 10,201
  Mean    : 5.3856
  Std Dev : 5.2968
  Min     : 0
  25%     : 2.00
  Median  : 4.00
  75%     : 8.00
  Max     : 51
  Skew    : 1.7968
  Kurt    : 4.7914

DESCRIPTIVE STATISTICS — Home Player Aces per Match
  Count   : 10,201
  Mean    : 2.7113
  Std Dev : 3.2536
  Min     : 0
  25%     : 0.00
  Median  : 2.00
  75%     : 4.00
  Max     : 30
  Skew    : 2.0329
  Kurt    : 5.8235

DESCRIPTIVE STATISTICS — Away Player Aces per Match
  Count   : 10,201
  Mean    : 2.6743
  Std Dev : 3.1924
  Min     : 0
  25%     : 0.00
  Median  : 2.00
  75%     : 4.00
  Max     : 39
  Skew    : 2.0904
  Kurt    : 6.9641
